## Human-AI Collaboration: Information Gain (IG)

This framework evaluates the LLM not just as a standalone tool, but as a **collaborator**. We use the **Information Gain (IG)** metric from the 2025 paper *"Smarter Metrics That Separate Hype from Trust"* to distinguish between models that are merely accurate on paper and those that are truly reliable in a team.

### 1. The Metric Formula
$$IG = \text{AI-Help} - \lambda \cdot \text{AI-Harm}$$

*   **AI-Help**: Instances where the human was originally **wrong**, but the AI led them to the **correct** answer.
*   **AI-Harm**: Instances where the human was originally **correct**, but the AI misled them into a **wrong** answer (Induced Harm).
*   **$\lambda$ (Safety Penalty)**: We use $\lambda = 2$. This penalizes "Induced Harm" twice as heavily as we reward "Help," prioritizing trust over simple performance.

### 2. Simulation Setup
Since we don't have real human feedback in this loop, we simulate the interaction:
1.  **Human Baseline ($h_0$)**: A simulated human makes a guess with a base accuracy (default 60%).
2.  **Team Decision ($h_1$)**: The human decides whether to trust the AI's answer or stick with their own, based on a **Trust Rate** (default 70%).

---
The following cells define these simulation functions and run the consolidated evaluation pipeline.

In [5]:
import pandas as pd
import numpy as np
import random
import os
from typing import Dict
from dotenv import load_dotenv

# .envrc format is 'export KEY="VALUE"', so we use a custom function to load it 
# since standard load_dotenv might not handle 'export' and the specific format correctly.
def load_envrc(filepath):
    if os.path.exists(filepath):
        with open(filepath, 'r') as f:
            for line in f:
                line = line.strip()
                if line.startswith('export '):
                    # Remove 'export ' and handle both 'KEY="VALUE"' and 'KEY: "VALUE"' formats
                    part = line[7:]
                    if '=' in part:
                        key, value = part.split('=', 1)
                    elif ': ' in part:
                        key, value = part.split(': ', 1)
                    else:
                        continue
                    
                    # Strip quotes from key and value
                    key = key.strip().strip('"').strip("'")
                    value = value.strip().strip('"').strip("'")
                    os.environ[key] = value

load_envrc('.envrc')
print("OPENAI_API_KEY loaded:", "OPENAI_API_KEY" in os.environ)


OPENAI_API_KEY loaded: True


In [9]:
# OpenAI API call
from openai import OpenAI
import os

# Ensure you have set your OPENAI_API_KEY environment variable
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

def run_llm(prompt):
    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {"role": "user", "content": prompt}
        ],
        temperature=0
    )
    return response.choices[0].message.content

In [1]:
import pandas as pd
eval_df = pd.read_csv('data/dataset_math_with_llm_output.csv')

In [6]:
eval_df

,question,answer,LLM_output
0,Jungkook is the 5th place. Find the number of ...,"If Jungkook is in 5th place, then 4 people cro...",To determine the number of people who crossed ...
1,A number divided by 10 is 6. Yoongi got the re...,"Let's call the certain number ""x"". According t...","To solve the problem, we need to find the numb..."
2,Dongju selects a piece of paper with a number ...,To find the second smallest and third smallest...,"To solve this problem, we need to determine al..."
3,"You wanted to subtract 46 from a number, but y...",If you accidentally subtracted 59 instead of 4...,"To solve this problem, let's break it down ste..."
4,The length of one span of Jinseo is about 12 c...,If one span of Jinseo is about 12 centimeters ...,"To solve the problem, we need to determine the..."
5,The owner of the snack shop said that for a sp...,"To receive the most sweets, Haneul should make...","To solve this problem, we need to form the lar..."
6,"For the natural number A, the quotient of A di...","To find the value of A, we can use the formula...","To solve the problem, we need to use the infor..."
7,How many diagonals can you draw in a decagon?,A decagon is a polygon with 10 sides. To find ...,To determine the number of diagonals in a deca...
8,What is the difference between the largest num...,"To find the largest number, we should arrange ...","To solve this problem, we need to form the lar..."
9,Find the sum of all multiples of 9 that are le...,To find the sum of all multiples of 9 that are...,To find the sum of all multiples of 9 that are...


In [10]:
import re

def parse_answer(text):
    """Simple parser to extract numerical or short answers."""
    if text is None: return None
    # Look for patterns like "The answer is X" or "\boxed{X}"
    text = str(text).replace(',', '')
    match = re.search(r'(?:[Aa]nswer is|boxed\{)([^.}]+)\b', text)
    if match:
        result = match.group(1).strip()
        try:
            return float(result) if '.' in result or result.isdigit() else result
        except:
            return result
    # Fallback: take the last word
    return text.split()[-1].strip('.')

def simulate_human(y, accuracy=0.6):
    """Simulates initial human decision with a specific accuracy."""
    if random.random() < accuracy:
        return y
    return "Incorrect Option" # Placeholder for a wrong answer

def simulate_decision(h0, a, trust_rate=0.7):
    """Simulates team decision where human trusts AI with some probability."""
    # A common model: Human switches to AI if AI is present, with some trust factor
    if random.random() < trust_rate:
        return a
    return h0

def compute_metrics(h0, a, h1, y, lmbda=2.0):
    """
    Computes Human-AI metrics including Information Gain (IG).
    Formula: IG = AI-Help - lambda * AI-Harm
    """
    is_correct_h0 = 1 if str(h0).lower() == str(y).lower() else 0
    is_correct_a = 1 if str(a).lower() == str(y).lower() else 0
    is_correct_h1 = 1 if str(h1).lower() == str(y).lower() else 0
    
    # AI-Help: Human wrong -> Team right
    ai_help = 1 if (is_correct_h0 == 0 and is_correct_h1 == 1) else 0
    
    # AI-Harm: Human right -> Team wrong
    ai_harm = 1 if (is_correct_h0 == 1 and is_correct_h1 == 0) else 0
    
    # Information Gain (IG)
    ig = ai_help - (lmbda * ai_harm)
    
    return {
        'h0_acc': is_correct_h0,
        'ai_acc': is_correct_a,
        'team_acc': is_correct_h1,
        'ai_help': ai_help,
        'ai_harm': ai_harm,
        'information_gain': ig
    }

In [ ]:
# Run the evaluation pipeline using the defined metrics and IG formula

results = []
metrics = None # For compatibility with later cells

# Using eval_df (which already contains LLM_output from previous steps or file loading)
for _, row in eval_df.iterrows():
    question = row['question']
    y = row['answer']
    
    # 1. Get LLM output
    # We use the existing column to avoid NameErrors and redundant API calls
    if 'LLM_output' in row:
        llm_output = row['LLM_output']
    elif 'llm_output' in row:
        llm_output = row['llm_output']
    else:
        # Fallback if no output is present: define prompt and run
        def build_prompt_inline(q): return f"Q: {q}\nSolve the problem, think step by step and give your answer."
        llm_output = run_llm(build_prompt_inline(question))
    
    # 2. Parse answer (from the previous cell)
    a = parse_answer(llm_output)

    if a is None:
        continue

    # 3. Simulate Human-AI interaction
    # We simulate these to calculate Information Gain (IG)
    h0 = simulate_human(y, accuracy=0.6) # Initial human guess
    h1 = simulate_decision(h0, a, trust_rate=0.7) # Team decision (Human + AI)

    # 4. Compute all metrics including Information Gain (IG = Help - 2*Harm)
    metrics = compute_metrics(h0, a, h1, y, lmbda=2.0)
    
    # Add metadata for reporting
    metrics.update({
        'question': question,
        'ground_truth': y,
        'llm_output': llm_output,
        'parsed_prediction': a
    })
    
    results.append(metrics)

results_df = pd.DataFrame(results)

,h0_acc,ai_acc,team_acc,ai_help,ai_harm,information_gain,question,ground_truth,llm_output,parsed_prediction
0,1,0,1,0,0,0.0,Jungkook is the 5th place. Find the number of ...,"If Jungkook is in 5th place, then 4 people cro...",To determine the number of people who crossed ...,4
1,1,0,1,0,0,0.0,A number divided by 10 is 6. Yoongi got the re...,"Let's call the certain number ""x"". According t...","To solve the problem, we need to find the numb...",45
2,1,0,1,0,0,0.0,Dongju selects a piece of paper with a number ...,To find the second smallest and third smallest...,"To solve this problem, we need to determine al...",804.0
3,0,0,0,0,0,0.0,"You wanted to subtract 46 from a number, but y...",If you accidentally subtracted 59 instead of 4...,"To solve this problem, let's break it down ste...",56
4,1,0,1,0,0,0.0,The length of one span of Jinseo is about 12 c...,If one span of Jinseo is about 12 centimeters ...,"To solve the problem, we need to determine the...",centimeters


In [12]:
results_df

,h0_acc,ai_acc,team_acc,ai_help,ai_harm,information_gain,question,ground_truth,llm_output,parsed_prediction
0,1,0,1,0,0,0.0,Jungkook is the 5th place. Find the number of ...,"If Jungkook is in 5th place, then 4 people cro...",To determine the number of people who crossed ...,4
1,1,0,1,0,0,0.0,A number divided by 10 is 6. Yoongi got the re...,"Let's call the certain number ""x"". According t...","To solve the problem, we need to find the numb...",45
2,1,0,1,0,0,0.0,Dongju selects a piece of paper with a number ...,To find the second smallest and third smallest...,"To solve this problem, we need to determine al...",804.0
3,0,0,0,0,0,0.0,"You wanted to subtract 46 from a number, but y...",If you accidentally subtracted 59 instead of 4...,"To solve this problem, let's break it down ste...",56
4,1,0,1,0,0,0.0,The length of one span of Jinseo is about 12 c...,If one span of Jinseo is about 12 centimeters ...,"To solve the problem, we need to determine the...",centimeters
5,0,0,0,0,0,0.0,The owner of the snack shop said that for a sp...,"To receive the most sweets, Haneul should make...","To solve this problem, we need to form the lar...",sweets
6,1,0,0,0,1,-2.0,"For the natural number A, the quotient of A di...","To find the value of A, we can use the formula...","To solve the problem, we need to use the infor...",59.0
7,1,0,0,0,1,-2.0,How many diagonals can you draw in a decagon?,A decagon is a polygon with 10 sides. To find ...,To determine the number of diagonals in a deca...,diagonals
8,1,0,0,0,1,-2.0,What is the difference between the largest num...,"To find the largest number, we should arrange ...","To solve this problem, we need to form the lar...",6497.0
9,1,0,0,0,1,-2.0,Find the sum of all multiples of 9 that are le...,To find the sum of all multiples of 9 that are...,To find the sum of all multiples of 9 that are...,324.0


In [12]:
metrics


{'h0_acc': 1,
 'ai_acc': 0,
 'team_acc': 1,
 'ai_help': 0,
 'ai_harm': 0,
 'information_gain': 0.0,
 'question': "Taehyung is trying to get to his grandmother's house, which is 300 kilometers (km) away on a motorcycle at 60 kilometers (km) per hour. Find how far Taehyung needs to go when 2 hours have passed since he left.",
 'ground_truth': "If Taehyung is traveling at a speed of 60 kilometers per hour, then in 2 hours he would have traveled:\n\n60 km/hour * 2 hours = 120 kilometers\n\nSince his grandmother's house is 300 kilometers away, and he has already traveled 120 kilometers, the remaining distance he needs to travel is:\n\n300 kilometers - 120 kilometers = 180 kilometers\n\nSo, Taehyung still needs to go 180 kilometers to reach his grandmother's house.",
 'llm_output': "To solve this problem, we need to determine how far Taehyung has traveled after 2 hours and then subtract that distance from the total distance to his grandmother's house.\n\n1. **Calculate the distance traveled 

In [ ]:
# Calculate overall performance
summary_metrics = results_df.mean()

print("--- Overall Human-AI Evaluation Metrics ---")
print(f"Human Baseline Accuracy (h0): {summary_metrics['h0_acc']:.2%}")
print(f"AI Solo Accuracy (a):         {summary_metrics['ai_acc']:.2%}")
print(f"Team Accuracy (h1):           {summary_metrics['team_acc']:.2%}")
print("-" * 40)
print(f"Information Gain (IG):        {summary_metrics['information_gain']:.4f}")
print("  (IG = AI_Help - 2 * AI_Harm)")
print("-" * 40)
print(f"AI Help (Wrong -> Right):     {summary_metrics['ai_help']:.2%}")
print(f"AI Harm (Right -> Wrong):     {summary_metrics['ai_harm']:.2%}")

"To solve the problem, we need to determine the length of the shorter side of the bookshelf in centimeters, given that it measures about two spans of Jinseo's hand.\n\n1. **Understand the measurement unit**: We are told that one span of Jinseo's hand is approximately 12 centimeters.\n\n2. **Determine the total length in spans**: The shorter side of the bookshelf is measured to be about two spans.\n\n3. **Calculate the length in centimeters**: Since one span is 12 centimeters, two spans would be:\n   \\[\n   2 \\text{ spans} \\times 12 \\text{ cm/span} = 24 \\text{ cm}\n   \\]\n\nTherefore, the length of the shorter side of the bookshelf is approximately 24 centimeters."